In [1]:
import os
from typing import Dict, List
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain.chat_models import ChatOpenAI
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain.retrievers.self_query.base import SelfQueryRetriever
from langchain.retrievers.self_query.base import Document
from langchain.chains.query_constructor.base import AttributeInfo
from langchain.smith import RunEvalConfig, run_on_dataset
from langsmith import Client
from langchain.callbacks.tracers.langchain import LangChainTracer
from langchain.callbacks.manager import CallbackManager
from langchain_community.embeddings import HuggingFaceEmbeddings

import pandas as pd
import chromadb

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
# LangSmith 클라이언트 및 트레이서 설정
client = Client()
tracer = LangChainTracer(project_name="SMALLTALK2REC")
callback_manager = CallbackManager([tracer])

In [4]:
# 영화 데이터 사용
df = pd.read_csv("../../preprocessed_movie_sample.csv")

# ChromaDB 클라이언트 초기화
client = chromadb.Client()

# 영화 데이터를 위한 컬렉션 생성
collection = client.create_collection("movies")


In [5]:
model_name = "BAAI/bge-m3"
model_kwargs = {'device': 'mps'}
encode_kwargs = {'normalize_embeddings': True}
huggingface_ef = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

/var/folders/dn/qq5pnzjj1c98r1dbp_pgns1w0000gn/T/ipykernel_63744/1850178229.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  huggingface_ef = HuggingFaceEmbeddings(
/Users/visuworks/Documents/00_project/SmallTalk2Rec_Exp/my_env/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


In [6]:
# 임베딩 생성 
df['plot_embedding'] = df['plot'].apply(lambda text: huggingface_ef.embed_query(text))
### comment 점수와 코멘트로 새로운 형태로 생성 필요

for index, row in df.iterrows():
    collection.add(
        documents=[row['plot']],  # 필드별 정보
        embeddings=[row['plot_embedding']],  # 필드별 임베딩
        metadatas=[{
            'title': row['title'],
            'director': row['director'],
            'screenwriter' : row['screenwriter'],
            'rating': row['rating'],
            'actors': row['actors'],
            'genres': row['genres'],
            'countries' : row['countries'],
            'running_time' : row['running_time'],
            'adult' : row['adult']
        }],
        ids=[str(index)]  # 고유 ID 설정
    )

In [7]:
metadata_field_info = [
    AttributeInfo(
        name="title",
        description="The title of the movie.",
        type="string",
    ),
    AttributeInfo(
        name="director",
        description="A list of the movie's directors.",
        type="list",
    ),
    AttributeInfo(
        name="screenwriter",
        description="A list of the movie's screenwriters.",
        type="list",
    ),
    AttributeInfo(
        name="plot",
        description="A short summary of the movie's plot.",
        type="string",
    ),
    AttributeInfo(
        name="rating",
        description="The average rating given to the movie. Range 0.0 ~ 5.0",
        type="float",
    ),
    AttributeInfo(
        name="rating_count",
        description="The total number of ratings the movie has received.",
        type="float",
    ),
    AttributeInfo(
        name="actors",
        description="A list of the movie's main actors.",
        type="list",
    ),
    AttributeInfo(
        name="genres",
        description="A list of the genres.",
        type="list",
    ),
    AttributeInfo(
        name="countries",
        description="A list of the countries where the movie was produced.",
        type="list",
    ),
    AttributeInfo(
        name="audience",
        description="Cumulative audience",
        type="float",
    ),
    AttributeInfo(
        name="running_time",
        description="The running time of the movie in minutes.",
        type="float",
    ),
    AttributeInfo(
        name="adult",
        description="A flag indicating whether the movie is for adults only.",
        type="float",
    ),
]
content_description = "plot and reviews of the movie"


In [8]:
llm = ChatOpenAI(temperature=0, 
               model='gpt-4o-mini-2024-07-18')

/var/folders/dn/qq5pnzjj1c98r1dbp_pgns1w0000gn/T/ipykernel_63744/2669306141.py:1: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(temperature=0,


In [9]:
vectorstore = Chroma(
    collection_name="movies",
    client=client,
    embedding_function=huggingface_ef
)

/var/folders/dn/qq5pnzjj1c98r1dbp_pgns1w0000gn/T/ipykernel_63744/2464781284.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [10]:


retriever = SelfQueryRetriever.from_llm(
            llm=llm,
            vectorstore=vectorstore,
            document_contents=content_description,
            metadata_field_info=metadata_field_info,
            verbose=True
        )

In [12]:
retriever.invoke('ENTP 영화추천해줘')

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


[Document(metadata={'actors': "['태런 애저튼', '콜린 퍼스']", 'adult': 1.0, 'countries': "['영국', '미국']", 'director': "['매튜 본']", 'genres': "['액션', '범죄', '코미디', '모험', '스릴러']", 'rating': 4.1, 'running_time': 128.0, 'screenwriter': "['제인 골드먼', '매슈 본']", 'title': '킹스맨: 시크릿 에이전트'}, page_content='세상에서 가장 위험한 면접이 시작된다!\n\n높은 IQ, 주니어 체조대회 2년 연속 우승! 그러나 학교 중퇴, 해병대 중도 하차. 동네 패싸움에 직장은 가져본 적도 없이 별볼일 없는 루저로 낙인 찍혔던 ‘그’가 ‘젠틀맨 스파이’로 전격 스카우트 됐다!\n\n전설적 베테랑 요원 해리 하트(콜린 퍼스)는 경찰서에 구치된 에그시(태런 애거튼)를 구제한다. 탁월한 잠재력을 알아본 그는 에그시를 전설적 국제 비밀정보기구 ‘킹스맨’ 면접에 참여시킨다. 아버지 또한 ‘킹스맨’의 촉망 받는 요원이었으나 해리 하트를 살리기 위해 죽었다는 사실을 알게 된 에그시. 목숨을 앗아갈 만큼 위험천만한 훈련을 통과해야 하는 킹스맨 후보들. 최종 멤버 발탁을 눈 앞에 둔 에그시는 최고의 악당 발렌타인(사무엘 L. 잭슨)을 마주하게 되는데…'),
 Document(metadata={'actors': "['로니 델 카르멘', '레아 루이스', '쉴라 옴미', '마무두 아티', '웬디 맥클렌던 커비', '캐서린 오하라', '메이슨 베르트하이머', '조 페라']", 'adult': 0.0, 'countries': "['미국']", 'director': "['피터 손']", 'genres': "['애니메이션', '모험', '판타지', 'SF', '로맨스']", 'rating': 3.8, 'running_time': 109.0, 'screenwriter': "['존 호버그', '캣 리켈', '브렌다 쉬

In [12]:
class MovieRecommender:
    def __init__(self,chromadb,content_description):
        self.embeddings = OpenAIEmbeddings()
        self.vectorstore = chromadb.as_retriever()
        
        self.llm = ChatOpenAI(
            temperature=0,
            callback_manager=callback_manager,
            tags=["movie-recommendation"]
        )
        
        self.retriever = SelfQueryRetriever.from_llm(
            self.llm,
            self.vectorstore,
            document_contents=content_description,
            metadata_field_info=metadata_field_info,
            verbose=True
        )
        
        self.recommendation_template = """
        사용자의 MBTI와 현재 감정 상태를 바탕으로 가장 적합한 영화를 추천해주세요.

        사용자 정보:
        - MBTI: {mbti}
        - 현재 감정: {emotion}

        관련 영화 정보:
        {movie_info}

        다음 형식으로 추천해주세요:
        1. 추천 영화 제목
        2. 추천 이유 (MBTI와 감정 상태를 연관지어 설명)
        3. 영화의 주요 특징
        4. 시청 포인트

        응답은 친근하고 공감적인 톤으로 작성해주세요.
        """
        
        self.recommendation_prompt = PromptTemplate(
            input_variables=["mbti", "emotion", "movie_info"],
            template=self.recommendation_template
        )
        
        self.recommendation_chain = LLMChain(
            llm=self.llm,
            prompt=self.recommendation_prompt,
            callback_manager=callback_manager,
            tags=["recommendation-chain"]
        )

    def get_recommendations(self, mbti: str, emotion: str) -> tuple[str, str]:
        """영화 추천을 생성하고 추적합니다."""
        run = client.create_run(
            project_name="SMALLTALK2REC",
            name="movie_recommendation",
            inputs={
                "mbti": mbti,
                "emotion": emotion
            }
        )
        
        try:
            # 영화 검색
            query = f"Find movies suitable for {mbti} personality type and {emotion} emotional state"
            docs = self.retriever.get_relevant_documents(query)

            # 메타데이터 기록
            client.update_run(
                run.id,
                metadata={
                    "num_results": len(docs),
                    "query": query
                }
            )

            # 영화 정보 통합
            movie_info = "\n".join([
                f"제목: {doc.page_content}\n메타데이터: {doc.metadata}\n"
                for doc in docs[:3]
            ])

            # 추천 생성
            recommendation = self.recommendation_chain.run({
                "mbti": mbti,
                "emotion": emotion,
                "movie_info": movie_info
            })

            # 성공적인 실행 기록
            client.update_run(
                run.id,
                outputs={"recommendation": recommendation},
                status="completed"
            )

            return recommendation, run.id

        except Exception as e:
            # 에러 발생 시 기록
            client.update_run(
                run.id,
                error=str(e),
                status="failed"
            )
            raise e
